# Maral NMT: General-Purpose Colab Trainer

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rcst-thesis/maral-onnx-nmt/blob/main/notebooks/maral_nmt_colab.ipynb)

Train a sequence-to-sequence model for any language pair from a two-column
parallel corpus. The notebook uses a single TPU with PyTorch/XLA, a shared
SentencePiece tokenizer, fixed-shape training, and CPU TorchScript/ONNX export.

Before running, select **Runtime → Change runtime type → TPU**.


## Install & Imports

In [ ]:
%%capture
# PyTorch and PyTorch/XLA are supplied as a compatible pair by the Colab TPU runtime.
# Do not pip-install or upgrade torch in this notebook.
!pip install -q sentencepiece pandas matplotlib tqdm sacrebleu onnx onnxruntime


In [ ]:
import os

# These must be set before the XLA device is initialized.
os.environ.setdefault("PJRT_DEVICE", "TPU")
os.environ.setdefault("XLA_USE_BF16", "1")

import re, io, math, random, time, warnings
import json as _json
from pathlib import Path
from typing import List, Optional, Tuple

import requests
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
import sentencepiece as spm
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from tqdm.auto import tqdm

try:
    import torch_xla
    import torch_xla.core.xla_model as xm
    from torch_xla.distributed.parallel_loader import MpDeviceLoader
except ImportError as exc:
    raise RuntimeError(
        "PyTorch/XLA is unavailable. In Colab select Runtime > Change runtime type "
        "> TPU (v6e), restart the runtime, then run the notebook from the top."
    ) from exc

warnings.filterwarnings("ignore")
torch.manual_seed(42)
random.seed(42)
np.random.seed(42)

DEVICE = (
    torch_xla.device()
    if hasattr(torch_xla, "device")
    else xm.xla_device()
)

if DEVICE.type != "xla":
    raise RuntimeError(f"Expected an XLA TPU device, but received {DEVICE}.")

print(f"Device       : {DEVICE}")
print(f"PyTorch      : {torch.__version__}")
print(f"PyTorch/XLA  : {torch_xla.__version__}")
print("TPU precision: BF16")


## Global Config

In [ ]:
# @title Project and training configuration
SOURCE_LANGUAGE = "English" # @param {type:"string"}
TARGET_LANGUAGE = "Target language" # @param {type:"string"}
DATASET_PATH = "/content/drive/MyDrive/datasets/parallel_corpus.tsv" # @param {type:"string"}
SOURCE_COLUMN = "source" # @param {type:"string"}
TARGET_COLUMN = "target" # @param {type:"string"}
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/MaralNMT/models" # @param {type:"string"}

CFG = {
    "vocab_size": 8_000,
    "spm_type": "bpe",
    "character_coverage": 0.9995,
    "d_model": 256,
    "n_heads": 8,
    "d_ff": 512,
    "n_enc_layers": 6,
    "n_dec_layers": 2,
    "dropout": 0.1,
    "max_len": 128,
    "block_size": 2,
    "batch_size": 256,
    "num_epochs": 1000,
    "lr": 2e-4,
    "warmup_steps": 8_000,
    "label_smoothing": 0.1,
    "weight_decay": 1e-2,
    "clip_grad": 1.0,
    "val_every": 1,
    "save_best": True,
    "early_stop_patience": 10,
    "grad_accum": 1,
    "token_mask_rate": 0.15,
    "beam_size": 4,
    "max_decode_len": 80,
    "len_penalty": 0.6,
}

for directory in ("data", "models", "checkpoints"):
    Path(directory).mkdir(exist_ok=True)

print(f"Language pair: {SOURCE_LANGUAGE} → {TARGET_LANGUAGE}")
print(
    f"TPU config: batch_size={CFG['batch_size']}, "
    f"fixed_length={CFG['max_len']}, BF16=True"
)


## Load Dataset


In [ ]:
from google.colab import drive

if DATASET_PATH.startswith("/content/drive/") or DRIVE_OUTPUT_DIR.startswith("/content/drive/"):
    drive.mount("/content/drive")


In [ ]:
def _clean(df):
    """Normalize a configured parallel corpus to source/target columns."""
    if SOURCE_COLUMN == TARGET_COLUMN:
        raise ValueError("SOURCE_COLUMN and TARGET_COLUMN must be different.")

    missing = [
        column for column in (SOURCE_COLUMN, TARGET_COLUMN)
        if column not in df.columns
    ]
    if missing:
        raise ValueError(
            f"Missing columns {missing}. Available columns: {df.columns.tolist()}"
        )

    clean_df = df[[SOURCE_COLUMN, TARGET_COLUMN]].rename(
        columns={SOURCE_COLUMN: "source", TARGET_COLUMN: "target"}
    ).dropna()
    clean_df = clean_df.astype(str).apply(lambda column: column.str.strip())
    clean_df = clean_df[
        clean_df["source"].str.len().between(3, 400)
        & clean_df["target"].str.len().between(3, 400)
    ]
    return clean_df.drop_duplicates().reset_index(drop=True)


def _write_tsv(df, path):
    lines = df["source"].str.cat(df["target"], sep="\t")
    Path(path).write_text("\n".join(lines) + "\n", encoding="utf-8")


def load_and_prepare_dataset(filepath):
    """Load, validate, shuffle, and split a two-column TSV corpus."""
    path = Path(filepath)
    if not path.exists():
        raise FileNotFoundError(
            f"Dataset not found: {path}. Update DATASET_PATH in the config cell."
        )

    full_df = pd.read_csv(path, sep="\t", on_bad_lines="skip")
    clean_df = _clean(full_df).sample(frac=1, random_state=42).reset_index(drop=True)
    if len(clean_df) < 2:
        raise ValueError("The dataset must contain at least two valid sentence pairs.")

    train_end = max(1, int(len(clean_df) * 0.8))
    train_df = clean_df.iloc[:train_end]
    val_df = clean_df.iloc[train_end:]

    _write_tsv(train_df, "data/train.tsv")
    _write_tsv(val_df, "data/val.tsv")
    corpus = pd.concat([train_df["source"], train_df["target"]], ignore_index=True)
    Path("data/corpus.txt").write_text(
        "\n".join(corpus) + "\n", encoding="utf-8"
    )

    print(f"Valid pairs: {len(clean_df):,}")
    print(f"Train: {len(train_df):,} | Validation: {len(val_df):,}")
    return train_df, val_df


train_df, val_df = load_and_prepare_dataset(DATASET_PATH)
display(train_df.head())


## SentencePiece Tokeniser (trained from scratch)

In [ ]:
class BilingualTokenizer:
    PAD, UNK, BOS, EOS = 0, 1, 2, 3

    def __init__(self, vocab_size=8_000):
        self.vocab_size = vocab_size
        self.prefix = "models/tokenizer"
        self.sp = None

    def train(self):
        model_path = Path(f"{self.prefix}.model")
        if model_path.exists():
            print("Tokenizer already exists — loading.")
            self.load()
            return

        spm.SentencePieceTrainer.train(
            input="data/corpus.txt",
            model_prefix=self.prefix,
            vocab_size=self.vocab_size,
            character_coverage=CFG["character_coverage"],
            model_type=CFG["spm_type"],
            pad_id=self.PAD,
            unk_id=self.UNK,
            bos_id=self.BOS,
            eos_id=self.EOS,
            num_threads=os.cpu_count() or 4,
            shuffle_input_sentence=True,
        )
        self.load()
        print(f"Vocabulary size: {self.sp.get_piece_size()}")

    def load(self):
        self.sp = spm.SentencePieceProcessor(model_file=f"{self.prefix}.model")

    def encode(self, text, add_bos=True, add_eos=True):
        ids = self.sp.encode_as_ids(text)
        return ([self.BOS] if add_bos else []) + ids + ([self.EOS] if add_eos else [])

    def decode(self, ids):
        if isinstance(ids, torch.Tensor):
            ids = ids.cpu().tolist()
        ids = [token for token in ids if token not in (self.PAD, self.BOS, self.EOS)]
        return self.sp.decode_ids(ids)

    def pad_id(self): return self.PAD
    def bos_id(self): return self.BOS
    def eos_id(self): return self.EOS
    def __len__(self): return self.sp.get_piece_size()


tokenizer = BilingualTokenizer(CFG["vocab_size"])
tokenizer.train()

for label, sample in (
    (SOURCE_LANGUAGE, train_df.iloc[0]["source"]),
    (TARGET_LANGUAGE, train_df.iloc[0]["target"]),
):
    encoded = tokenizer.encode(sample)
    print(f"{label}: {sample}")
    print(f"Round trip: {tokenizer.decode(encoded)}")


## Dataset & DataLoader

In [ ]:
from torch.utils.data import Dataset, DataLoader
import torch
from pathlib import Path


class TranslationDataset(Dataset):
    """Pre-tokenize every sentence pair once on the CPU."""

    def __init__(self, tsv_path: str, tok: BilingualTokenizer, max_len: int):
        self.data = []
        print(f"  Pre-tokenising {Path(tsv_path).name} …", end="", flush=True)

        if not Path(tsv_path).exists():
            print(f"\nWARNING: {tsv_path} not found. Returning empty dataset.")
            return

        with open(tsv_path, encoding="utf-8") as file:
            for line in file:
                parts = line.rstrip("\n").split("\t")
                if len(parts) != 2:
                    continue

                src = tok.encode(parts[0], add_bos=True, add_eos=True)[:max_len]
                tgt = tok.encode(parts[1], add_bos=True, add_eos=True)[:max_len]

                self.data.append(
                    (
                        torch.tensor(src, dtype=torch.long),
                        torch.tensor(tgt, dtype=torch.long),
                    )
                )

        print(f" {len(self.data):,} pairs ready.")

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
        return self.data[index]


def fixed_shape_collate(batch):
    """Pad every batch to one static XLA shape: [B, CFG['max_len']]."""

    batch_size = len(batch)
    max_len = CFG["max_len"]

    src_batch = torch.full(
        (batch_size, max_len),
        tokenizer.PAD,
        dtype=torch.long,
    )
    tgt_batch = torch.full(
        (batch_size, max_len),
        tokenizer.PAD,
        dtype=torch.long,
    )

    for row, (src, tgt) in enumerate(batch):
        src_length = min(src.numel(), max_len)
        tgt_length = min(tgt.numel(), max_len)
        src_batch[row, :src_length] = src[:src_length]
        tgt_batch[row, :tgt_length] = tgt[:tgt_length]

    return src_batch, tgt_batch


train_ds = TranslationDataset("data/train.tsv", tokenizer, CFG["max_len"])
val_ds = TranslationDataset("data/val.tsv", tokenizer, CFG["max_len"])

# Fixed batch sizes prevent a separate XLA compilation for the final batch.
# Validation drops at most batch_size-1 examples; add a padded-example mask if
# every validation row must be retained in a later revision.
_num_workers = min(2, os.cpu_count() or 1)

train_loader = DataLoader(
    train_ds,
    batch_size=CFG["batch_size"],
    shuffle=True,
    collate_fn=fixed_shape_collate,
    num_workers=_num_workers,
    persistent_workers=(_num_workers > 0),
    prefetch_factor=(2 if _num_workers > 0 else None),
    pin_memory=False,
    drop_last=True,
)

val_loader = DataLoader(
    val_ds,
    batch_size=CFG["batch_size"],
    shuffle=False,
    collate_fn=fixed_shape_collate,
    num_workers=_num_workers,
    persistent_workers=(_num_workers > 0),
    prefetch_factor=(2 if _num_workers > 0 else None),
    pin_memory=False,
    drop_last=True,
)

if len(train_loader) == 0 or len(val_loader) == 0:
    raise RuntimeError(
        "The fixed TPU batch size is larger than a dataset split. "
        "Reduce CFG['batch_size'] and rerun this cell."
    )

train_device_loader = MpDeviceLoader(train_loader, DEVICE)
val_device_loader = MpDeviceLoader(val_loader, DEVICE)

print(f"\nTrain batches : {len(train_loader):,}")
print(f"Val batches   : {len(val_loader):,}")
print(
    f"Static shapes : src=[{CFG['batch_size']}, {CFG['max_len']}], "
    f"tgt=[{CFG['batch_size']}, {CFG['max_len']}]"
)


## Model Building Blocks


### Utils

In [ ]:
class RMSNorm(nn.Module):
    """Root Mean Square Layer Normalization.
    Simpler than LayerNorm – no mean centering – and used in the AttnRes paper."""
    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()
        self.eps    = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: [..., dim]
        rms = x.norm(2, dim=-1, keepdim=True) * (x.size(-1) ** -0.5)
        return x / (rms + self.eps) * self.weight


class SinusoidalPositionalEncoding(nn.Module):
    """Fixed sin/cos positional encoding – no learned parameters."""
    def __init__(self, d_model: int, max_len: int = 5000, dropout: float = 0.1):
        super().__init__()
        self.drop = nn.Dropout(dropout)
        pe        = torch.zeros(max_len, d_model)
        pos       = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term  = torch.exp(torch.arange(0, d_model, 2, dtype=torch.float)
                              * (-math.log(10_000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div_term)
        pe[:, 1::2] = torch.cos(pos * div_term)
        self.register_buffer("pe", pe.unsqueeze(0))  # [1, max_len, d_model]

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.drop(x + self.pe[:, : x.size(1)])


In [ ]:
class TokenDrop(nn.Module):
    """Token Drop regularisation (Lample et al. / BERT-MLM inspired).

    Replaces entire source-token embeddings, encouraging the encoder to use context.\n\n    Scheduling
    ----------
    ``rate`` is reduced by 0.5× every ``anneal_epochs`` epochs so that late
    training can still learn fine-grained lexical detail.

    Args
    ----
    vocab_size  : int   Size of shared BPE vocabulary.
    d_model     : int   Embedding dimension (used to create mask embedding).
    rate        : float Fraction of tokens replaced per step (default 0.15).
    pad_id      : int   Padding token – never masked.
    bos_id      : int   BOS token – never masked.
    eos_id      : int   EOS token – never masked.
    """
    MASK_TOKEN_IDX = -1   # internal sentinel; replaced by learned embedding

    def __init__(self, vocab_size: int, d_model: int,
                 rate: float = 0.15,
                 pad_id: int = 0, bos_id: int = 2, eos_id: int = 3):
        super().__init__()
        self.rate    = rate
        self.pad_id  = pad_id
        self.bos_id  = bos_id
        self.eos_id  = eos_id
        # Learned mask embedding (one d_model vector, no grad through token ids)
        self.mask_emb = nn.Parameter(torch.zeros(d_model))

    def forward(self, src_ids: torch.Tensor,
                embeddings: torch.Tensor) -> torch.Tensor:
        """
        Parameters
        ----------
        src_ids    : [B, T]       Raw token IDs (used to find special tokens).
        embeddings : [B, T, D]    Already-embedded source tokens.

        Returns
        -------
        [B, T, D]  Embeddings with ~rate fraction of non-special positions
                   replaced by the learned mask vector.
        """
        if not self.training or self.rate <= 0.0:
            return embeddings

        # Build mask: True where we MAY replace (non-special tokens only)
        maskable = (
            (src_ids != self.pad_id) &
            (src_ids != self.bos_id) &
            (src_ids != self.eos_id)
        )  # [B, T]

        # Sample Bernoulli noise for positions that are maskable
        drop_mask = (
            torch.bernoulli(
                torch.full_like(maskable, self.rate, dtype=torch.float)
            ).bool() & maskable
        )  # [B, T]

        # Replace selected positions with learned mask embedding
        mask_vec  = self.mask_emb.to(embeddings.dtype)          # [D]
        drop_exp  = drop_mask.unsqueeze(-1).expand_as(embeddings)
        out       = torch.where(drop_exp, mask_vec.expand_as(embeddings),
                                embeddings)
        return out


### Block Attention Residuals (AttnRes)


In [ ]:
class BlockAttnResOp(nn.Module):
    """
    A single Block Attention Residual operation.

    Given a list of completed block representations and the current
    intra-block partial sum, it returns a depth-wise attended hidden state
    that is then used as input to the next sub-layer.

    Args
    ----
    d_model : int  Hidden dimension.
    """
    def __init__(self, d_model: int):
        super().__init__()
        # Learned pseudo-query – one scalar weight per hidden dimension.
        # Initialised to zero so initial attention is uniform (safe start).
        self.w        = nn.Parameter(torch.zeros(d_model))
        self.key_norm = RMSNorm(d_model)   # Prevents large-mag blocks dominating

    def forward(self,
                block_reps: List[torch.Tensor],
                partial: torch.Tensor) -> torch.Tensor:
        """
        Parameters
        ----------
        block_reps : list of [B, T, D] – completed block summaries b_0 … b_{n-1}
        partial    : [B, T, D]         – current intra-block running sum
        Returns
        -------
        [B, T, D]  Depth-wise softmax-attended representation.
        """
        # OPTIMIZED: pre-stack once instead of building a Python list then stacking
        sources = block_reps + [partial]           # len = N+1
        V       = torch.stack(sources, dim=0)      # [N+1, B, T, D]
        K       = self.key_norm(V)                 # Normalize keys

        # Attention logits: w · K  (broadcast over B, T)
        logits  = (K * self.w).sum(-1)             # [N+1, B, T]
        weights = F.softmax(logits, dim=0)         # [N+1, B, T]

        # Weighted sum over depth
        out = (weights.unsqueeze(-1) * V).sum(0)   # [B, T, D]
        return out


### Encoder Layer & Encoder

In [ ]:
class NMTEncoderLayer(nn.Module):
    """
    One encoder layer with two AttnRes operations (pre-attention, pre-FFN)
    and standard intra-block residual accumulation.
    """
    def __init__(self, d_model: int, n_heads: int, d_ff: int, dropout: float):
        super().__init__()
        # Block Attention Residuals – one per sub-layer
        self.ar_attn = BlockAttnResOp(d_model)
        self.ar_ffn  = BlockAttnResOp(d_model)
        # Self-attention
        self.self_attn = nn.MultiheadAttention(d_model, n_heads,
                                               dropout=dropout,
                                               batch_first=True)
        # Feed-forward network
        self.ffn1 = nn.Linear(d_model, d_ff)
        self.ffn2 = nn.Linear(d_ff, d_model)
        # PreNorm (applied to the AttnRes output before each sub-layer)
        self.norm1 = RMSNorm(d_model)
        self.norm2 = RMSNorm(d_model)
        self.drop  = nn.Dropout(dropout)

    def forward(self,
                block_reps: List[torch.Tensor],
                partial: torch.Tensor,
                src_pad_mask: Optional[torch.Tensor] = None) -> torch.Tensor:
        """
        block_reps : completed block summaries from previous blocks
        partial    : intra-block running sum so far (updated in-place)
        Returns    : updated partial [B, T, D]
        """
        # ── Self-attention ────────────────────────────────────────────────
        h        = self.ar_attn(block_reps, partial)   # depth-wise attended input
        h_normed = self.norm1(h)
        attn_out, _ = self.self_attn(h_normed, h_normed, h_normed,
                                     key_padding_mask=src_pad_mask,
                                     need_weights=False)
        partial = self.drop(partial) + self.drop(attn_out)  # residual dropout on both sides

        # ── Feed-forward network ──────────────────────────────────────────
        h        = self.ar_ffn(block_reps, partial)
        h_normed = self.norm2(h)
        # OPTIMIZED: approximate='tanh' GELU is ~2× faster with negligible accuracy loss
        ffn_out  = self.ffn2(self.drop(F.gelu(self.ffn1(h_normed), approximate='tanh')))
        partial  = self.drop(partial) + self.drop(ffn_out)  # residual dropout on both sides

        return partial


class NMTEncoder(nn.Module):
    """
    Full encoder stack.
    Layers are grouped into blocks of size `block_size`.
    The encoder returns the **full sequence** of hidden states [B, T, D]
    (not a scalar block summary) so the decoder's cross-attention can attend
    over all source positions.
    """
    def __init__(self, cfg: dict):
        super().__init__()
        d  = cfg["d_model"]
        self.layers = nn.ModuleList([
            NMTEncoderLayer(d, cfg["n_heads"], cfg["d_ff"], cfg["dropout"])
            for _ in range(cfg["n_enc_layers"])
        ])
        self.block_size = cfg["block_size"]
        self.final_norm = RMSNorm(d)

    def forward(self,
                x: torch.Tensor,
                src_pad_mask: Optional[torch.Tensor] = None) -> torch.Tensor:
        """
        x            : [B, T, D]  embedded + positionally encoded source
        src_pad_mask : [B, T]     True where position is padding
        Returns      : [B, T, D]  encoder hidden states
        """
        block_reps: List[torch.Tensor] = [x]
        partial = x.clone()   # Intra-block running sum, starts as embedding

        for i, layer in enumerate(self.layers):
            partial = layer(block_reps, partial, src_pad_mask)

            if (i + 1) % self.block_size == 0 and (i + 1) < len(self.layers):
                block_reps.append(partial)
                partial = torch.zeros_like(x)  # Fresh intra-block accumulator

        return self.final_norm(partial)


### Decoder Layer & Decoder

In [ ]:
class NMTDecoderLayer(nn.Module):
    """
    One decoder layer:
      1. AttnRes-attended masked self-attention (autoregressive)
      2. Standard cross-attention to encoder output
      3. AttnRes-attended FFN
    """
    def __init__(self, d_model: int, n_heads: int, d_ff: int, dropout: float):
        super().__init__()
        # AttnRes ops for self-attn and FFN portions
        self.ar_self = BlockAttnResOp(d_model)
        self.ar_ffn  = BlockAttnResOp(d_model)

        self.self_attn  = nn.MultiheadAttention(d_model, n_heads,
                                                dropout=dropout, batch_first=True)
        self.cross_attn = nn.MultiheadAttention(d_model, n_heads,
                                                dropout=dropout, batch_first=True)
        self.ffn1 = nn.Linear(d_model, d_ff)
        self.ffn2 = nn.Linear(d_ff, d_model)
        self.norm1 = RMSNorm(d_model)   # before self-attn
        self.norm2 = RMSNorm(d_model)   # before cross-attn
        self.norm3 = RMSNorm(d_model)   # before ffn
        self.drop  = nn.Dropout(dropout)

    def forward(self,
                block_reps: List[torch.Tensor],
                partial: torch.Tensor,
                enc_out: torch.Tensor,
                causal_mask: torch.Tensor,
                tgt_pad_mask: Optional[torch.Tensor] = None,
                src_pad_mask: Optional[torch.Tensor] = None) -> torch.Tensor:

        # ── 1. Masked self-attention (AttnRes input) ──────────────────────
        h         = self.ar_self(block_reps, partial)
        h_normed  = self.norm1(h)
        sa_out, _ = self.self_attn(h_normed, h_normed, h_normed,
                                   attn_mask=causal_mask,
                                   key_padding_mask=tgt_pad_mask,
                                   need_weights=False)
        partial = self.drop(partial) + self.drop(sa_out)  # residual dropout

        # ── 2. Cross-attention to encoder (standard, no AttnRes) ──────────
        ca_out, _ = self.cross_attn(self.norm2(partial), enc_out, enc_out,
                                    key_padding_mask=src_pad_mask,
                                    need_weights=False)
        partial = self.drop(partial) + self.drop(ca_out)  # residual dropout

        # ── 3. Feed-forward (AttnRes input) ──────────────────────────────
        h       = self.ar_ffn(block_reps, partial)
        # OPTIMIZED: approximate GELU (same as encoder)
        ffn_out = self.ffn2(self.drop(F.gelu(self.ffn1(self.norm3(h)), approximate='tanh')))
        partial = self.drop(partial) + self.drop(ffn_out)  # residual dropout

        return partial


class NMTDecoder(nn.Module):
    """Full decoder stack with Block AttnRes."""
    def __init__(self, cfg: dict):
        super().__init__()
        d = cfg["d_model"]
        self.layers = nn.ModuleList([
            NMTDecoderLayer(d, cfg["n_heads"], cfg["d_ff"], cfg["dropout"])
            for _ in range(cfg["n_dec_layers"])
        ])
        self.block_size = cfg["block_size"]
        self.final_norm = RMSNorm(d)
        self.out_proj   = nn.Linear(d, cfg["vocab_size"])

    def forward(self,
                x: torch.Tensor,
                enc_out: torch.Tensor,
                causal_mask: torch.Tensor,
                tgt_pad_mask: Optional[torch.Tensor] = None,
                src_pad_mask: Optional[torch.Tensor] = None) -> torch.Tensor:
        """
        x           : [B, T_tgt, D] embedded target sequence
        enc_out     : [B, T_src, D] encoder output
        causal_mask : [T_tgt, T_tgt] upper-triangular -inf mask
        Returns     : [B, T_tgt, vocab_size] logits
        """
        block_reps: List[torch.Tensor] = [x]
        partial = x.clone()

        for i, layer in enumerate(self.layers):
            partial = layer(block_reps, partial, enc_out,
                            causal_mask, tgt_pad_mask, src_pad_mask)

            if (i + 1) % self.block_size == 0 and (i + 1) < len(self.layers):
                block_reps.append(partial)
                partial = torch.zeros_like(x)

        return self.out_proj(self.final_norm(partial))


### Full Seq2Seq NMT Model

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
from torch.distributions import Categorical

class Seq2SeqNMT(nn.Module):
    def __init__(self, cfg: dict, vocab_size: int):
        super().__init__()
        d = cfg["d_model"]
        self.src_embed = nn.Embedding(vocab_size, d, padding_idx=0)
        self.tgt_embed = nn.Embedding(vocab_size, d, padding_idx=0)
        self.pos_enc   = SinusoidalPositionalEncoding(d, max_len=512, dropout=cfg["dropout"])
        self.encoder   = NMTEncoder(cfg)
        self.decoder   = NMTDecoder(cfg)
        self.cfg       = cfg
        self._emb_scale = math.sqrt(d)
        self.decoder.out_proj.weight = self.tgt_embed.weight
        # Token Drop: replaces whole source tokens (not neurons) to build
        # context-aware encoder representations for WSD.
        self.token_drop = TokenDrop(
            vocab_size = vocab_size,
            d_model    = cfg["d_model"],
            rate       = cfg.get("token_mask_rate", 0.15),
            pad_id     = 0, bos_id = 2, eos_id = 3,
        )
        self._init_weights()

    def _init_weights(self):
        for p in self.parameters():
            if p.dim() > 1: nn.init.xavier_uniform_(p)
        for m in self.modules():
            if isinstance(m, BlockAttnResOp): nn.init.zeros_(m.w)

    @staticmethod
    def make_pad_mask(seq: torch.Tensor) -> torch.Tensor:
        return seq == 0

    @staticmethod
    def make_causal_mask(size: int, device: torch.device) -> torch.Tensor:
        return torch.triu(torch.full((size, size), float("-inf"), device=device), diagonal=1)

    def _embed_src(self, src):
        emb = self.src_embed(src) * self._emb_scale
        emb = self.token_drop(src, emb)   # Token Drop (no-op at eval)
        return self.pos_enc(emb)

    def _embed_tgt(self, tgt):
        return self.pos_enc(self.tgt_embed(tgt) * self._emb_scale)

    def forward(self, src: torch.Tensor, tgt: torch.Tensor) -> torch.Tensor:
        """Standard forward pass (used at inference and for BLEU eval)."""
        src_pad = self.make_pad_mask(src)
        tgt_pad = self.make_pad_mask(tgt)
        causal  = self.make_causal_mask(tgt.size(1), tgt.device)
        enc_out = self.encoder(self._embed_src(src), src_pad)
        return self.decoder(self._embed_tgt(tgt), enc_out, causal, tgt_pad, src_pad)


    @torch.no_grad()
    def translate(self, src, bos_id, eos_id, beam_size=4, max_len=150, len_penalty=0.6, entropy_threshold=2.0, wide_beam_size=8, return_entropy=False):
        device, src_pad = src.device, self.make_pad_mask(src)
        enc_out = self.encoder(self._embed_src(src), src_pad)
        enc_exp, pad_exp = enc_out.expand(beam_size, -1, -1).contiguous(), src_pad.expand(beam_size, -1).contiguous()
        beams, finished, step_entropy, causal_cache = [(0.0, [bos_id])], [], [], {}
        for _ in range(max_len):
            active = [(s, seq) for s, seq in beams if seq[-1] != eos_id]
            finished.extend([(s, seq) for s, seq in beams if seq[-1] == eos_id])
            if not active: break
            cur_len, n_active = len(active[0][1]), len(active)
            tgt_batch = torch.tensor([seq for _, seq in active], dtype=torch.long, device=device)
            if cur_len not in causal_cache: causal_cache[cur_len] = self.make_causal_mask(cur_len, device)
            logits_batch = self.decoder(self._embed_tgt(tgt_batch), enc_exp[:n_active], causal_cache[cur_len], None, pad_exp[:n_active])
            log_p_batch = F.log_softmax(logits_batch[:, -1, :], dim=-1)
            candidates, step_H = [], []
            for i, (score, seq) in enumerate(active):
                log_p = log_p_batch[i]
                h_bits = Categorical(probs=log_p.exp()).entropy().item() / math.log(2)
                step_H.append(h_bits)
                k = wide_beam_size if h_bits > entropy_threshold else beam_size
                val, idx = log_p.topk(k)
                for lp, tok in zip(val.tolist(), idx.tolist()): candidates.append((score + lp, seq + [tok]))
            if step_H: step_entropy.append(max(step_H))
            candidates.sort(key=lambda x: x[0] / (len(x[1]) ** len_penalty), reverse=True)
            beams = candidates[:beam_size]
            if all(s[-1] == eos_id for _, s in beams): break
        finished = finished or beams
        finished.sort(key=lambda x: x[0] / (len(x[1]) ** len_penalty), reverse=True)
        best_ids = torch.tensor(finished[0][1], device=device)
        return (best_ids, step_entropy) if return_entropy else best_ids

model = Seq2SeqNMT(CFG, len(tokenizer)).to(DEVICE)
print("Model initialized in eager mode (torch.compile disabled to prevent InductorError).")

## Loss, Optimiser & Learning Rate Schedule

The loss and AdamW optimizer remain ordinary PyTorch objects. The training loop
uses `xm.optimizer_step()` so XLA can execute and synchronize each update.


In [ ]:
# Translation loss
class LabelSmoothingLoss(nn.Module):
    """
    Token-level cross-entropy with label smoothing.

    Padding tokens are excluded from the loss.
    """

    def __init__(
        self,
        pad_id: int,
        smoothing: float = 0.1,
    ):
        super().__init__()

        if not 0.0 <= smoothing < 1.0:
            raise ValueError(
                "Label smoothing must be between 0 and 1."
            )

        self.pad_id = pad_id
        self.smoothing = smoothing

    def forward(
        self,
        logits: torch.Tensor,
        targets: torch.Tensor,
    ) -> torch.Tensor:
        """
        logits:  [batch, target_length, vocabulary]
        targets: [batch, target_length]
        """
        vocab_size = logits.size(-1)

        return F.cross_entropy(
            logits.reshape(-1, vocab_size),
            targets.reshape(-1),
            ignore_index=self.pad_id,
            label_smoothing=self.smoothing,
        )


criterion = LabelSmoothingLoss(
    pad_id=tokenizer.PAD,
    smoothing=CFG["label_smoothing"],
)


# Optimizer

# Avoid weight decay on bias and normalization parameters.
decay_parameters = []
no_decay_parameters = []

for parameter_name, parameter in model.named_parameters():
    if not parameter.requires_grad:
        continue

    if (
        parameter.ndim < 2
        or parameter_name.endswith(".bias")
        or "norm" in parameter_name.lower()
    ):
        no_decay_parameters.append(parameter)
    else:
        decay_parameters.append(parameter)


optimizer = AdamW(
    [
        {
            "params": decay_parameters,
            "weight_decay": CFG.get("weight_decay", 1e-2),
        },
        {
            "params": no_decay_parameters,
            "weight_decay": 0.0,
        },
    ],
    lr=CFG["lr"],
    betas=(0.9, 0.98),
    eps=1e-9,
)


# Warmup + cosine learning-rate scheduler

class WarmupCosineScheduler:
    """
    Linear warmup followed by cosine decay.

    The learning rate is prepared for the first optimizer update during
    initialization, preventing the first batch from using the full LR.
    """

    def __init__(
        self,
        optimizer,
        warmup_steps: int,
        total_steps: int,
        min_lr: float = 1e-6,
    ):
        if total_steps <= 0:
            raise ValueError("total_steps must be greater than zero.")

        self.optimizer = optimizer
        self.warmup_steps = max(0, int(warmup_steps))
        self.total_steps = max(
            int(total_steps),
            self.warmup_steps + 1,
        )
        self.min_lr = float(min_lr)

        self.base_lrs = [
            group["lr"]
            for group in optimizer.param_groups
        ]

        # Number of optimizer updates already completed.
        self._step = 0

        # Prepare LR for optimizer update number 1.
        self._set_lr_for_update(update_number=1)

    def _lr_for_update(
        self,
        base_lr: float,
        update_number: int,
    ) -> float:
        update_number = max(1, update_number)

        if (
            self.warmup_steps > 0
            and update_number <= self.warmup_steps
        ):
            return (
                base_lr
                * update_number
                / self.warmup_steps
            )

        decay_steps = max(
            1,
            self.total_steps - self.warmup_steps,
        )

        progress = (
            update_number - self.warmup_steps
        ) / decay_steps

        progress = min(max(progress, 0.0), 1.0)

        return (
            self.min_lr
            + 0.5
            * (base_lr - self.min_lr)
            * (1.0 + math.cos(math.pi * progress))
        )

    def _set_lr_for_update(
        self,
        update_number: int,
    ):
        for group, base_lr in zip(
            self.optimizer.param_groups,
            self.base_lrs,
        ):
            group["lr"] = self._lr_for_update(
                base_lr,
                update_number,
            )

    def step(self):
        """
        Call after optimizer.step().

        Prepares the learning rate for the next optimizer update.
        """
        self._step += 1
        self._set_lr_for_update(
            update_number=self._step + 1
        )

        return self.get_last_lr()

    def get_last_lr(self):
        return [
            group["lr"]
            for group in self.optimizer.param_groups
        ]

    def state_dict(self):
        return {
            "step": self._step,
            "warmup_steps": self.warmup_steps,
            "total_steps": self.total_steps,
            "min_lr": self.min_lr,
            "base_lrs": self.base_lrs,
        }

    def load_state_dict(self, state_dict):
        self._step = int(state_dict["step"])
        self.warmup_steps = int(
            state_dict["warmup_steps"]
        )
        self.total_steps = int(
            state_dict["total_steps"]
        )
        self.min_lr = float(state_dict["min_lr"])
        self.base_lrs = list(state_dict["base_lrs"])

        self._set_lr_for_update(
            update_number=self._step + 1
        )


# Number of optimizer updates, not number of batches.
grad_accum = max(1, CFG.get("grad_accum", 1))

updates_per_epoch = math.ceil(
    len(train_loader) / grad_accum
)

total_steps = (
    CFG["num_epochs"]
    * updates_per_epoch
)

warmup_steps = min(
    CFG["warmup_steps"],
    max(1, total_steps - 1),
)

scheduler = WarmupCosineScheduler(
    optimizer=optimizer,
    warmup_steps=warmup_steps,
    total_steps=total_steps,
    min_lr=1e-6,
)


print(
    f"Optimizer       : AdamW\n"
    f"Base LR         : {CFG['lr']:.2e}\n"
    f"Initial warmup LR: "
    f"{scheduler.get_last_lr()[0]:.2e}\n"
    f"Weight decay    : "
    f"{CFG.get('weight_decay', 1e-2)}\n"
    f"Label smoothing : "
    f"{CFG['label_smoothing']}\n"
    f"Updates/epoch   : "
    f"{updates_per_epoch:,}\n"
    f"Warmup updates  : "
    f"{warmup_steps:,}\n"
    f"Total updates   : "
    f"{total_steps:,}"
)

## Training Loop

In [ ]:
# ================================================================
# Single-device TPU v6e training loop
#
# Important XLA rule: no .item(), .cpu(), or metric text refresh per batch.
# Metrics stay as XLA tensors and transfer to the host once per epoch.
# ================================================================

from tqdm.notebook import tqdm as NotebookTqdm
from pathlib import Path
import time
import torch
import torch.nn as nn


Path("checkpoints").mkdir(parents=True, exist_ok=True)


def run_xla_epoch(
    model,
    device_loader,
    number_of_batches,
    criterion,
    optimizer=None,
    scheduler=None,
    clip=1.0,
    train=True,
    grad_accum=1,
):
    """Run one static-shape TPU epoch with one host metric read at the end."""

    model.train(train)
    grad_accum = max(1, int(grad_accum))

    total_loss_sum = torch.zeros((), device=DEVICE, dtype=torch.float32)
    total_correct = torch.zeros((), device=DEVICE, dtype=torch.float32)
    total_confidence = torch.zeros((), device=DEVICE, dtype=torch.float32)
    total_tokens = torch.zeros((), device=DEVICE, dtype=torch.float32)

    final_group_size = number_of_batches % grad_accum
    if final_group_size == 0:
        final_group_size = grad_accum
    final_group_start = number_of_batches - final_group_size

    phase = "Training" if train else "Validating"
    progress = NotebookTqdm(
        device_loader,
        total=number_of_batches,
        desc=phase,
        leave=False,
        unit="batch",
        mininterval=0.5,
    )

    if train:
        if optimizer is None:
            raise ValueError("optimizer is required when train=True")
        optimizer.zero_grad(set_to_none=True)

    context = torch.enable_grad() if train else torch.no_grad()

    try:
        with context:
            for step_index, (src, tgt) in enumerate(progress):
                dec_in = tgt[:, :-1]
                dec_tgt = tgt[:, 1:]

                divisor = (
                    final_group_size
                    if step_index >= final_group_start
                    else grad_accum
                )

                # XLA_USE_BF16 handles TPU matrix precision; CUDA AMP is not used.
                logits = model(src, dec_in)
                loss = criterion(logits, dec_tgt)

                if train:
                    (loss / divisor).backward()

                    update_boundary = ((step_index + 1) % grad_accum == 0)
                    last_batch = ((step_index + 1) == number_of_batches)

                    if update_boundary or last_batch:
                        nn.utils.clip_grad_norm_(model.parameters(), max_norm=clip)
                        xm.optimizer_step(optimizer, barrier=False)
                        optimizer.zero_grad(set_to_none=True)

                        if scheduler is not None:
                            scheduler.step()

                # All metric operations remain on the TPU.
                detached_logits = logits.detach()
                max_logits, predictions = detached_logits.max(dim=-1)
                log_normalizer = torch.logsumexp(detached_logits, dim=-1)
                token_confidence = torch.exp(max_logits - log_normalizer)
                valid_mask = dec_tgt.ne(tokenizer.PAD)
                valid_tokens = valid_mask.sum().to(torch.float32)

                total_tokens = total_tokens + valid_tokens
                total_correct = total_correct + (
                    predictions.eq(dec_tgt) & valid_mask
                ).sum().to(torch.float32)
                # Multiplication keeps the tensor shape static. Avoid masked_select,
                # whose data-dependent output length is a poor fit for XLA.
                total_confidence = total_confidence + (
                    token_confidence * valid_mask.to(token_confidence.dtype)
                ).sum().to(torch.float32)
                total_loss_sum = total_loss_sum + loss.detach().to(torch.float32) * valid_tokens

                del logits, loss, detached_logits, token_confidence

    finally:
        progress.close()

    # One synchronization and one small device-to-host transfer per epoch.
    xm.mark_step()
    xm.wait_device_ops()

    metric_tensor = torch.stack(
        [total_loss_sum, total_correct, total_confidence, total_tokens]
    ).cpu()

    loss_sum, correct_sum, confidence_sum, token_count = metric_tensor.tolist()
    token_count = max(token_count, 1.0)

    return (
        loss_sum / token_count,
        100.0 * correct_sum / token_count,
        100.0 * confidence_sum / token_count,
    )


history = {
    "epochs": [],
    "train_loss": [],
    "val_loss": [],
    "train_acc": [],
    "val_acc": [],
    "train_conf": [],
    "val_conf": [],
    "lr": [],
}

best_val_loss = float("inf")
best_val_acc = 0.0
best_val_conf = 0.0
best_epoch = 0
last_completed_epoch = 0
patience_counter = 0
stopped_early = False

grad_accum = max(1, int(CFG.get("grad_accum", 1)))
patience = max(1, int(CFG.get("early_stop_patience", 10)))

print(
    f"Starting TPU training on {DEVICE} | BF16 | "
    f"batch={CFG['batch_size']} | fixed_len={CFG['max_len']}"
)
print("The first training and validation passes include XLA compilation time.")

epoch_bar = NotebookTqdm(
    range(1, CFG["num_epochs"] + 1),
    desc="TPU epochs",
    leave=True,
    unit="epoch",
)

try:
    for epoch in epoch_bar:
        epoch_start = time.time()

        train_loss, train_acc, train_conf = run_xla_epoch(
            model=model,
            device_loader=train_device_loader,
            number_of_batches=len(train_loader),
            criterion=criterion,
            optimizer=optimizer,
            scheduler=scheduler,
            clip=CFG["clip_grad"],
            train=True,
            grad_accum=grad_accum,
        )

        val_loss, val_acc, val_conf = run_xla_epoch(
            model=model,
            device_loader=val_device_loader,
            number_of_batches=len(val_loader),
            criterion=criterion,
            train=False,
            grad_accum=1,
        )

        elapsed = time.time() - epoch_start
        current_lr = optimizer.param_groups[0]["lr"]
        last_completed_epoch = epoch

        history["epochs"].append(epoch)
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)
        history["train_conf"].append(train_conf)
        history["val_conf"].append(val_conf)
        history["lr"].append(current_lr)

        checkpoint_status = "unchanged"

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_val_acc = val_acc
            best_val_conf = val_conf
            best_epoch = epoch
            patience_counter = 0
            checkpoint_status = "saved"

            xm.save(
                {
                    "epoch": epoch,
                    "state_dict": model.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(),
                    "scheduler_state_dict": (
                        scheduler.state_dict()
                        if scheduler is not None
                        and hasattr(scheduler, "state_dict")
                        else None
                    ),
                    "train_loss": train_loss,
                    "train_acc": train_acc,
                    "train_conf": train_conf,
                    "val_loss": val_loss,
                    "val_acc": val_acc,
                    "val_conf": val_conf,
                    "history": history,
                    "config": CFG.copy(),
                },
                "checkpoints/best.pt",
            )
        else:
            patience_counter += 1
            checkpoint_status = f"wait {patience_counter}/{patience}"

        epoch_bar.set_description(f"TPU epoch {epoch}/{CFG['num_epochs']}")
        epoch_bar.set_postfix(
            train_loss=f"{train_loss:.4f}",
            train_acc=f"{train_acc:.2f}%",
            train_conf=f"{train_conf:.2f}%",
            val_loss=f"{val_loss:.4f}",
            val_acc=f"{val_acc:.2f}%",
            val_conf=f"{val_conf:.2f}%",
            lr=f"{current_lr:.2e}",
            checkpoint=checkpoint_status,
            sec=f"{elapsed:.1f}",
            refresh=True,
        )

        if patience_counter >= patience:
            stopped_early = True
            epoch_bar.set_description(f"Early stopped at epoch {epoch}")
            break

finally:
    epoch_bar.close()

print("\nTPU training complete.")
print(f"Last completed epoch: {last_completed_epoch}")

if best_epoch > 0:
    print(
        f"Best epoch: {best_epoch} | val_loss={best_val_loss:.4f} | "
        f"val_acc={best_val_acc:.2f}% | val_conf={best_val_conf:.2f}%"
    )
    print("Best checkpoint: checkpoints/best.pt")

if stopped_early:
    print(f"Early stopping triggered after {patience} non-improving epochs.")


In [ ]:
# ── Save history to JSON (modified training logic persistence) ──
with open('checkpoints/history.json', 'w') as f:
    _json.dump(history, f)
print("History persisted to checkpoints/history.json")

## Loss Curves

In [ ]:
fig = plt.figure(figsize=(14, 10))
fig.suptitle(f"{SOURCE_LANGUAGE} → {TARGET_LANGUAGE} NMT",
             fontsize=14, fontweight="bold", y=1.01)

gs = gridspec.GridSpec(2, 2, figure=fig, hspace=0.40, wspace=0.35)

epochs_x = history["epochs"]

# ── Top-left: Train & Val loss ─────────────────────────
ax1 = fig.add_subplot(gs[0, 0])
ax1.plot(epochs_x, history["train_loss"], color="#2196F3",
         linewidth=2, label="Train loss")
ax1.plot(epochs_x, history["val_loss"],   color="#FF5722",
         linewidth=2, linestyle="--", label="Val loss")
ax1.set_xlabel("Epoch"); ax1.set_ylabel("Loss (label-smoothed CE)")
ax1.set_title("Train vs Validation Loss")
ax1.legend(); ax1.grid(alpha=0.3)

# ── Top-right: Learning rate schedule ──────────────────────
ax2 = fig.add_subplot(gs[0, 1])
ax2.plot(epochs_x, history["lr"], color="#4CAF50", linewidth=2)
ax2.set_xlabel("Epoch"); ax2.set_ylabel("Learning rate")
ax2.set_title("Learning Rate Schedule (Warmup + Cosine)")
ax2.ticklabel_format(axis="y", style="sci", scilimits=(0, 0))
ax2.grid(alpha=0.3)

# ── Bottom-left: Perplexity ─────────────────────────
ax3 = fig.add_subplot(gs[1, 0])
train_ppl = [math.exp(min(l, 10)) for l in history["train_loss"]]
val_ppl   = [math.exp(min(l, 10)) for l in history["val_loss"]]
ax3.plot(epochs_x, train_ppl, color="#2196F3", linewidth=2, label="Train PPL")
ax3.plot(epochs_x, val_ppl,   color="#FF5722", linewidth=2,
         linestyle="--", label="Val PPL")
ax3.set_xlabel("Epoch"); ax3.set_ylabel("Perplexity")
ax3.set_title("Perplexity (lower = better)")
ax3.legend(); ax3.grid(alpha=0.3)

# ── Bottom-right: Val loss delta (convergence speed) ───────
ax4 = fig.add_subplot(gs[1, 1])
if len(history["val_loss"]) > 1:
    deltas = [history["val_loss"][i] - history["val_loss"][i-1]
              for i in range(1, len(history["val_loss"]))]
    colors = ["#4CAF50" if d < 0 else "#F44336" for d in deltas]
    ax4.bar(epochs_x[1:], deltas, color=colors, alpha=0.75)
    ax4.axhline(0, color="black", linewidth=0.8, linestyle="--")
    ax4.set_xlabel("Epoch"); ax4.set_ylabel("Δ Val loss")
    ax4.set_title("Validation Loss Change per Epoch")
    ax4.grid(alpha=0.3, axis="y")
else:
    ax4.text(0.5, 0.5, "Need ≥ 2 epochs", ha="center", va="center",
             transform=ax4.transAxes)

plt.tight_layout()
plt.savefig("loss_curves.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → loss_curves.png")


## CPU Export: TorchScript and ONNX

The TPU checkpoint is loaded into a fresh CPU model before export. The exported
graph is the two-input teacher-forced forward pass `(src_tokens, tgt_tokens) ->
logits`. The mobile application must implement the autoregressive decode loop.


In [ ]:
import torch
import torch.nn as nn
import onnx
from pathlib import Path


CHECKPOINT_PATH = Path("checkpoints/best.pt")
TORCHSCRIPT_PATH = Path("nmt_model.pt")
ONNX_PATH = Path("nmt_model.onnx")

if not CHECKPOINT_PATH.exists():
    raise FileNotFoundError(
        "checkpoints/best.pt was not found. Complete training before export."
    )

checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=False,
)

export_config = checkpoint.get("config", CFG).copy()

cpu_model = Seq2SeqNMT(export_config, len(tokenizer)).cpu()
cpu_model.load_state_dict(checkpoint["state_dict"])
cpu_model.eval()


class NMTForwardExport(nn.Module):
    """Inference graph used by a mobile autoregressive decoding loop."""

    def __init__(self, base_model):
        super().__init__()
        self.base_model = base_model

    def forward(self, src_tokens, tgt_tokens):
        return self.base_model(src_tokens, tgt_tokens)


export_model = NMTForwardExport(cpu_model).eval()

dummy_src = torch.full(
    (1, export_config["max_len"]),
    tokenizer.PAD,
    dtype=torch.long,
)
dummy_src[0, 0] = tokenizer.BOS
dummy_src[0, 1] = tokenizer.EOS

dummy_tgt = torch.tensor(
    [[tokenizer.BOS, tokenizer.EOS]],
    dtype=torch.long,
)

with torch.no_grad():
    traced_model = torch.jit.trace(
        export_model,
        (dummy_src, dummy_tgt),
        strict=False,
    )
    traced_model.save(str(TORCHSCRIPT_PATH))

    torch.onnx.export(
        export_model,
        (dummy_src, dummy_tgt),
        str(ONNX_PATH),
        export_params=True,
        opset_version=17,
        do_constant_folding=True,
        input_names=["src_tokens", "tgt_tokens"],
        output_names=["logits"],
        dynamic_axes={
            "src_tokens": {0: "batch_size", 1: "source_length"},
            "tgt_tokens": {0: "batch_size", 1: "target_length"},
            "logits": {0: "batch_size", 1: "target_length"},
        },
    )

onnx_model = onnx.load(str(ONNX_PATH))
onnx.checker.check_model(onnx_model)

print(f"TorchScript saved: {TORCHSCRIPT_PATH}")
print(f"ONNX saved       : {ONNX_PATH}")
print("ONNX validation  : passed")


In [ ]:
# Save the checkpoint and deployment bundle to Google Drive.
import shutil
from datetime import datetime

required_files = [
    Path("checkpoints/best.pt"),
    Path("nmt_model.pt"),
    Path("nmt_model.onnx"),
    Path("models/tokenizer.model"),
]
missing = [str(path) for path in required_files if not path.exists()]
if missing:
    raise FileNotFoundError(
        "Run training and CPU export first. Missing: " + ", ".join(missing)
    )

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
drive_model_dir = Path(DRIVE_OUTPUT_DIR) / f"nmt_model_{timestamp}"
drive_model_dir.mkdir(parents=True, exist_ok=False)

for source in required_files + [
    Path("models/tokenizer.vocab"),
    Path("checkpoints/history.json"),
]:
    if source.exists():
        shutil.copy2(source, drive_model_dir / source.name)

with (drive_model_dir / "config.json").open("w", encoding="utf-8") as file:
    _json.dump(
        {
            **CFG,
            "source_language": SOURCE_LANGUAGE,
            "target_language": TARGET_LANGUAGE,
        },
        file,
        indent=2,
        ensure_ascii=False,
    )

print(f"Model bundle saved to: {drive_model_dir}")


## Validation Samples

Translate a few validation rows after CPU export. This is a qualitative smoke
test; use a held-out test set for reportable metrics.


In [ ]:
# @title Number of validation examples
NUM_EXAMPLES = 5 # @param {type:"integer"}

cpu_model.eval()
for row in val_df.head(max(0, NUM_EXAMPLES)).itertuples(index=False):
    source_ids = tokenizer.encode(row.source)
    source = torch.tensor([source_ids], dtype=torch.long)
    with torch.no_grad():
        output_ids = cpu_model.translate(
            source,
            bos_id=tokenizer.bos_id(),
            eos_id=tokenizer.eos_id(),
            beam_size=CFG["beam_size"],
            max_len=CFG["max_decode_len"],
            len_penalty=CFG["len_penalty"],
        )
    print(f"{SOURCE_LANGUAGE}: {row.source}")
    print(f"Prediction: {tokenizer.decode(output_ids)}")
    print(f"Reference: {row.target}\n")


In [ ]:
# @title Interactive translator
input_text = "Enter a sentence to translate." # @param {type:"string"}

cpu_model.eval()
with torch.no_grad():
    source = torch.tensor([tokenizer.encode(input_text)], dtype=torch.long)
    output_ids, entropy = cpu_model.translate(
        source,
        bos_id=tokenizer.bos_id(),
        eos_id=tokenizer.eos_id(),
        beam_size=CFG["beam_size"],
        max_len=CFG["max_decode_len"],
        len_penalty=CFG["len_penalty"],
        return_entropy=True,
    )

print(f"{SOURCE_LANGUAGE}: {input_text}")
print(f"{TARGET_LANGUAGE}: {tokenizer.decode(output_ids)}")
print(f"Confidence: {'LOW' if entropy and max(entropy) > 2.0 else 'HIGH'}")
